# Predictive multiplicity 실험 가이드

COMPAS 데이터 확인부터 XOR 모델 비교, 합성 데이터의 누락 피처 실험, COMPAS의 피처 추가 비교까지 순서대로 따라간다. 정확도는 각 실험에 사용한 표본에서 계산한다.

### 1. 파일 위치 확인

**하는 일과 이유**

현재 작업 폴더의 `data` 아래에서 파일을 찾는다. CSV를 읽기 전에 상대 경로를 점검하기 위한 단계다.

In [2]:
from pathlib import Path

for path in sorted(Path("data").rglob("*")):
    if path.is_file():
        print(path)

**결과 읽기**

출력이 비어 있으면 현재 위치 기준 `data` 아래에 파일을 찾지 못했다는 뜻이다. 다음 셀의 `../data`는 노트북 폴더를 기준으로 한 경로다.

### 2. COMPAS 데이터 불러오기

**하는 일과 이유**

CSV를 읽고 크기, 열 이름과 처음 다섯 행을 출력한다. 사용할 피처와 라벨이 있는지 확인하기 위한 단계다.

In [3]:
import pandas as pd

df = pd.read_csv("../data/compas_arrest_processed.csv")

print("num_rows, num_columns:", df.shape)
print("columns:", df.columns.tolist())
display(df.head())

num_rows, num_columns: (6172, 22)
columns: ['arrest', 'race_is_causasian', 'race_is_african_american', 'race_is_hispanic', 'race_is_other', 'age_leq_25', 'age_25_to_45', 'age_geq_46', 'female', 'n_priors_eq_0', 'n_priors_geq_1', 'n_priors_geq_2', 'n_priors_geq_5', 'n_juvenile_misdemeanors_eq_0', 'n_juvenile_misdemeanors_geq_1', 'n_juvenile_misdemeanors_geq_2', 'n_juvenile_misdemeanors_geq_5', 'n_juvenile_felonies_eq_0', 'n_juvenile_felonies_geq_1', 'n_juvenile_felonies_geq_2', 'n_juvenile_felonies_geq_5', 'charge_degree_eq_M']


,arrest,race_is_causasian,race_is_african_american,race_is_hispanic,race_is_other,age_leq_25,age_25_to_45,age_geq_46,female,n_priors_eq_0,...,n_priors_geq_5,n_juvenile_misdemeanors_eq_0,n_juvenile_misdemeanors_geq_1,n_juvenile_misdemeanors_geq_2,n_juvenile_misdemeanors_geq_5,n_juvenile_felonies_eq_0,n_juvenile_felonies_geq_1,n_juvenile_felonies_geq_2,n_juvenile_felonies_geq_5,charge_degree_eq_M
0,-1,0,0,0,1,0,0,1,0,1,...,0,1,0,0,0,1,0,0,0,0
1,1,0,1,0,0,0,1,0,0,1,...,0,1,0,0,0,1,0,0,0,0
2,1,0,1,0,0,1,0,0,0,0,...,0,1,0,0,0,1,0,0,0,0
3,-1,0,0,0,1,0,1,0,0,1,...,0,1,0,0,0,1,0,0,0,1
4,1,1,0,0,0,0,1,0,0,0,...,1,1,0,0,0,1,0,0,0,0


**결과 읽기**

표본은 6,172행과 22열이며, 예측할 라벨은 `arrest`다.

### 3. 열별 값 형식 확인

**하는 일과 이유**

결측값을 제외하고 각 열에 실제로 나타나는 값 중 최대 다섯 개를 출력한다. 피처와 라벨을 어떤 값으로 집계할지 확인한다.

In [ ]:
for col in df.columns:
    examples = df[col].dropna().unique()[:5].tolist()
    print(f"{col} → {examples}")

**결과 읽기**

피처는 0과 1, 라벨은 −1과 +1로 다룬다. 이 출력은 값의 예시이며 빈도는 다음 단계에서 확인한다.

### 4. 두 피처별 COMPAS 라벨 집계

**하는 일과 이유**

25세 이하 여부와 청소년기 중범죄 기록 1건 이상 여부를 각각 `x1`, `x2`로 두고 입력 조합별 라벨 수를 센다. 같은 입력이 서로 다른 정답을 갖는지 확인한다.

In [5]:
data = df[["age_leq_25", "n_juvenile_felonies_geq_1", "arrest"]].copy()
data.columns = ["x1", "x2", "y"]

groups = pd.MultiIndex.from_product([[0, 1], [0, 1]], names=["x1", "x2"])
counts = pd.crosstab([data["x1"], data["x2"]], data["y"])
counts = counts.reindex(index=groups, columns=[-1, 1], fill_value=0)
# counts["total_cnt_rows"] = counts.sum(axis=1)

display(counts)

y        -1     1
x1 x2            
0  0   2728  1953
   1     42   102
1  0    576   707
   1     17    47

**결과 읽기**

예를 들어 `(0,0)`의 −1은 2,728건, +1은 1,953건이다. 이 두 피처만 보면 같은 입력에 같은 예측을 해야 하므로 둘 다 맞힐 수는 없다.

### 5. XOR 데이터 만들기

**하는 일과 이유**

두 피처 값이 다르면 1, 같으면 0으로 붙인다. 피처와 라벨을 고정한 채 모델 클래스의 표현력을 비교하기 위한 데이터다.

In [6]:
import pandas as pd
from itertools import product

# 1단계에서 정한 네 문제와 XOR 정답
xor_data = pd.DataFrame({
    "x1": [0, 0, 1, 1],
    "x2": [0, 1, 0, 1],
    "y":  [0, 1, 1, 0],
})

# 입력 조합마다 정답 0과 정답 1이 각각 몇 개인지 집계
xor_counts = pd.crosstab([xor_data["x1"], xor_data["x2"]], xor_data["y"])
display(xor_counts)

y      0  1
x1 x2      
0  0   1  0
   1   0  1
1  0   0  1
   1   1  0

**결과 읽기**

라벨 1이 서로 마주 보는 두 꼭짓점에 있어 직선 하나로 정답을 나눌 수 없다.

### 6. 가능한 예측 패턴 채점

**하는 일과 이유**

네 입력 조합에 대한 16개 예측 패턴을 열거하고 라벨 집계표로 각각 채점한다. 깊이 2 이하 트리는 16개 모두, 선형 분류기는 두 대각선 패턴을 뺀 14개를 표현할 수 있다.

In [ ]:
# XOR 정답지를 기반으로 선형모델, 트리모델 각각에서 정답을 얼마나 맞추는지 보는 단계
def score_models(label_counts, negative, positive):
    # 예측값 네 개가 가리키는 문제 순서를 고정
    groups = [(0, 0), (0, 1), (1, 0), (1, 1)]
    # XOR 정답표에 들어있는 전체 사례수를 카운트
    total = int(label_counts[[negative, positive]].to_numpy().sum())

    # 직선 하나로 표현할 수 없는 두 대각선 패턴
    nonlinear_patterns = {
        (negative, positive, positive, negative),
        (positive, negative, negative, positive),
    }

    results = []

    for predictions in product([negative, positive], repeat=4):
        # 각 그룹에서 모델이 예측한 라벨과 실제 라벨이 같은 행 수를 더함
        correct = 0
        for group, prediction in zip(groups, predictions):
            correct += int(label_counts.at[group, prediction])

        model_classes = ["깊이 2 이하 트리"]
        if predictions not in nonlinear_patterns:
            model_classes.append("선형 분류기")

        for model_class in model_classes:
            results.append({
                "모델 클래스": model_class,
                "예측 (①, ②, ③, ④)": predictions,
                "맞힌 수": correct,
                "틀린 수": total - correct,
            })

    return pd.DataFrame(results)


xor_results = score_models(xor_counts, negative=0, positive=1)

**결과 읽기**

후보를 전부 채점해 두 클래스에서 공동 최적 패턴을 빠짐없이 찾을 수 있다.

### 7. XOR의 최적 패턴 비교

**하는 일과 이유**

클래스별로 최고 점수를 받은 패턴을 전부 남긴다. 모델 클래스가 바뀌면 최고 점수와 최적 패턴 수가 어떻게 달라지는지 확인한다.

In [ ]:
# 모델 클래스별로 가장 잘 맞춘 경우만을 가져옴
best_score = xor_results.groupby("모델 클래스")["맞힌 수"].transform("max")
xor_best = xor_results[xor_results["맞힌 수"] == best_score]

display(xor_best.sort_values("모델 클래스").reset_index(drop=True))

,모델 클래스,"예측 (①, ②, ③, ④)",맞힌 수,틀린 수
0,깊이 2 이하 트리,"(0, 1, 1, 0)",4,0
1,선형 분류기,"(0, 0, 1, 0)",3,1
2,선형 분류기,"(0, 1, 0, 0)",3,1
3,선형 분류기,"(0, 1, 1, 1)",3,1
4,선형 분류기,"(1, 1, 1, 0)",3,1


**결과 읽기**

선형 분류기는 3/4를 맞히는 패턴이 네 개다. 트리는 4/4를 맞히는 패턴이 하나다.

### 8. 같은 모델 비교를 COMPAS에 적용

**하는 일과 이유**

COMPAS 두 피처의 라벨 집계표를 같은 함수에 넣어 XOR와 같은 양상이 나타나는지 확인한다.

In [ ]:
# COMPAS에 기록된 실제 재채포 여부를 토대로 각 모델에서 답을 얼마나 잘 맞추는지 측정
compas_results = score_models(counts, negative=-1, positive=1)

best_score = compas_results.groupby("모델 클래스")["맞힌 수"].transform("max")
compas_best = compas_results[compas_results["맞힌 수"] == best_score]

display(compas_best.reset_index(drop=True))

,모델 클래스,"예측 (①, ②, ③, ④)",맞힌 수,틀린 수
0,깊이 2 이하 트리,"(-1, 1, 1, 1)",3584,2588
1,선형 분류기,"(-1, 1, 1, 1)",3584,2588


**결과 읽기**

두 클래스 모두 최적 패턴은 `(-1, 1, 1, 1)` 하나이고, 3,584건을 맞힌다(58.07%). 조합마다 다수 라벨이 하나로 정해져 오분류가 남아도 최적 예측은 하나다.

### 9. 선형 분류기의 예측 차이 표 만들기

**하는 일과 이유**

XOR 최적 패턴 네 개를 행에, 입력 조합을 열에 둔다. 첫 패턴 A를 기준으로 예측이 다른 칸에 `True`를 표시한다.

In [ ]:
# 선형 클래스에서 공동 1등을 한 답안 4개
patterns = xor_best.loc[
    xor_best["모델 클래스"] == "선형 분류기",
    "예측 (①, ②, ③, ④)"
]
print(patterns)

# 한 줄이 답안 하나, 한 열이 문제 하나
answers = pd.DataFrame(
    patterns.tolist(),
    columns=["① (0,0)", "② (0,1)", "③ (1,0)", "④ (1,1)"]
)
answers.index = ["기준 A", "B", "C", "D"]
display(answers)

# 기준 A와 답이 다른 칸에 True 표시
# .ne => 값이 다르니?를 물어보는거임 그러니까 True면 값 달러요.
different = answers.ne(answers.loc["기준 A"], axis="columns")
display(different)

5     (0, 0, 1, 0)
9     (0, 1, 0, 0)
14    (0, 1, 1, 1)
27    (1, 1, 1, 0)
Name: 예측 (①, ②, ③, ④), dtype: object


,"① (0,0)","② (0,1)","③ (1,0)","④ (1,1)"
기준 A,0,0,1,0
B,0,1,0,0
C,0,1,1,1
D,1,1,1,0


,"① (0,0)","② (0,1)","③ (1,0)","④ (1,1)"
기준 A,False,False,False,False
B,False,True,True,False
C,False,True,False,True
D,True,True,False,False


**결과 읽기**

예를 들어 B는 A와 `(0,1)`, `(1,0)`에서 다르다. 다음 셀에서 이 표를 두 지표로 요약한다.

### 10. 선형 분류기의 불일치 지표 계산

**하는 일과 이유**

열별로 다른 답이 한 번이라도 있는 사례의 비율을 ambiguity로 구한다. 후보 하나가 기준과 다르게 예측한 비율의 최댓값을 discrepancy로 구한다.

In [12]:
# 세로로 읽기: 문제마다 True가 하나라도 있는지 확인
changed_by_question = different.any(axis=0)
display(changed_by_question)

# 가로로 읽기: 모델마다 A와 다른 답이 몇 개인지 세기
changes_by_model = different.sum(axis=1)
display(changes_by_model)

n_questions = different.shape[1]  # XOR 문제 4개
ambiguity = changed_by_question.sum() / n_questions
discrepancy = changes_by_model.max() / n_questions

print(f"Ambiguity: {ambiguity:.0%}")
print(f"Discrepancy: {discrepancy:.0%}")

① (0,0)    True
② (0,1)    True
③ (1,0)    True
④ (1,1)    True
dtype: bool

기준 A    0
B       2
C       2
D       2
dtype: int64

Ambiguity: 100%
Discrepancy: 50%


**결과 읽기**

Ambiguity 100%는 네 사례 모두에서 기준과 다른 예측이 가능하다는 뜻이다. Discrepancy 50%는 기준과 가장 많이 다른 후보가 네 사례 중 두 사례에서 다르다는 뜻이다.

### 11. 트리의 예측 차이 표 만들기

**하는 일과 이유**

최적 트리 패턴을 가져와 첫 패턴을 기준으로 비교한다. 선형 분류기와 같은 방식으로 트리의 예측 차이를 확인한다.

In [13]:
tree_patterns = xor_best.loc[
    xor_best["모델 클래스"] == "깊이 2 이하 트리",
    "예측 (①, ②, ③, ④)"
]

tree_answers = pd.DataFrame(
    tree_patterns.tolist(),
    columns=["① (0,0)", "② (0,1)", "③ (1,0)", "④ (1,1)"]
)

# 첫 번째 최적 예측 패턴을 기준 A로 선택
tree_A = tree_answers.iloc[0]
tree_different = tree_answers.ne(tree_A, axis="columns")

display(tree_answers)
display(tree_different)

,"① (0,0)","② (0,1)","③ (1,0)","④ (1,1)"
0,0,1,1,0


,"① (0,0)","② (0,1)","③ (1,0)","④ (1,1)"
0,False,False,False,False


**결과 읽기**

최적 예측 패턴이 하나라 모든 칸이 `False`다.

### 12. 트리의 불일치 지표 계산

**하는 일과 이유**

같은 XOR 네 사례에서 선형 분류기에 적용한 공식을 트리에도 적용한다.

In [14]:
# 세로로 읽기: 각 문제에서 A와 다른 답이 한 번이라도 나왔나?
changed_by_question = tree_different.any(axis=0)

# 가로로 읽기: 각 답안에서 A와 다른 칸은 몇 개인가?
changes_by_model = tree_different.sum(axis=1)

n_questions = tree_different.shape[1]  # 4문제
ambiguity = changed_by_question.sum() / n_questions
discrepancy = changes_by_model.max() / n_questions

print("문제별로 한 번이라도 바뀌었나?", changed_by_question.tolist())
print("답안별로 바뀐 문제 수:", changes_by_model.tolist())
print(f"Ambiguity: {ambiguity:.0%}")
print(f"Discrepancy: {discrepancy:.0%}")

문제별로 한 번이라도 바뀌었나? [False, False, False, False]
답안별로 바뀐 문제 수: [0]
Ambiguity: 0%
Discrepancy: 0%


**결과 읽기**

Ambiguity와 discrepancy는 모두 0%다. 트리의 최적 패턴은 네 사례를 모두 맞히는 한 가지다.

## 누락 피처로 인한 예측 불일치

정답은 `y = x3`로 정한다. 두 조건에서 같은 다섯 행과 같은 라벨을 사용하고, 깊이 2 이하 트리에 제공하는 피처만 바꾼다. 모든 가능한 트리의 예측을 이 다섯 행에서 비교하고, 같은 예측은 하나로 센다. Ambiguity와 discrepancy의 분모는 모두 사례 수 5다.

### 13. 합성 데이터에서 정답 피처 숨기기

**하는 일과 이유**

다섯 사례의 라벨을 `y = x3`로 정한다. `tree_predictions`는 가능한 분기와 리프의 예측을 조합해 깊이 2 이하 트리의 예측을 열거한다. 라벨과 최대 깊이를 고정하고 `x3` 제공 여부만 바꾼다.

In [1]:
from itertools import product

# 각 행은 (x1, x2, x3)이며 정답은 x3를 그대로 사용한다.
feature_rows = [(0, 0, 0), (0, 0, 1), (0, 1, 1), (1, 0, 1), (1, 1, 0)]
labels = tuple(row[2] for row in feature_rows)


def tree_predictions(rows, feature_indices, depth):
    # 리프는 모든 행에 동일한 라벨을 예측한다.
    patterns = {(label,) * len(rows) for label in (0, 1)}
    if depth == 0:
        return patterns

    for feature in feature_indices:
        left = [row for row in rows if row[feature] == 0]
        right = [row for row in rows if row[feature] == 1]
        if not left or not right:
            continue
        left_patterns = tree_predictions(left, feature_indices, depth - 1)
        right_patterns = tree_predictions(right, feature_indices, depth - 1)
        for left_pattern, right_pattern in product(left_patterns, right_patterns):
            predictions = (iter(left_pattern), iter(right_pattern))
            patterns.add(tuple(next(predictions[row[feature]]) for row in rows))
    return patterns


for condition, features in [("x3 숨김", (0, 1)), ("x3 포함", (0, 1, 2))]:
    patterns = tree_predictions(feature_rows, features, depth=2)
    scores = {
        pattern: sum(prediction == label for prediction, label in zip(pattern, labels))
        for pattern in patterns
    }
    best_score = max(scores.values())
    best_patterns = sorted(pattern for pattern, score in scores.items() if score == best_score)
    reference = best_patterns[0]
    ambiguity = sum(len(set(column)) > 1 for column in zip(*best_patterns)) / len(labels)
    discrepancy = max(
        sum(prediction != baseline for prediction, baseline in zip(pattern, reference))
        for pattern in best_patterns
    ) / len(labels)

    print(f"{condition} | 최고 정확도: {best_score}/{len(labels)}")
    print(f"최적 예측 패턴 ({len(best_patterns)}개): {best_patterns}")
    print(f"Ambiguity: {ambiguity:.0%} | Discrepancy: {discrepancy:.0%}")

x3 숨김 | 최고 정확도: 4/5
최적 예측 패턴 (2개): [(0, 0, 1, 1, 0), (1, 1, 1, 1, 0)]
Ambiguity: 40% | Discrepancy: 40%
x3 포함 | 최고 정확도: 5/5
최적 예측 패턴 (1개): [(0, 1, 1, 1, 0)]
Ambiguity: 0% | Discrepancy: 0%


**결과 읽기**

`x3`를 숨기면 같은 입력을 가진 첫 두 사례를 둘 다 0이나 둘 다 1로 예측하는 패턴이 4/5로 공동 1등이다. 두 지표 모두 2/5인 40%다. `x3` 포함 시 5/5를 맞히는 패턴 하나가 최적이고 두 지표는 0%다.

## COMPAS에서 피처를 숨겼다가 다시 포함하기

기존 두 피처에 `n_priors_geq_1`(전과 1건 이상 여부)을 추가한다. 두 조건에서 같은 COMPAS 행과 관측 라벨을 사용하며, 깊이 2 이하인 모든 이진 의사결정 트리의 예측을 비교한다. 최고 정확도와의 차이가 0, 0.5, 1, 2%p 이내인 패턴을 각각 포함한다. 기준 모델은 각 조건에서 사전식으로 첫 번째인 최적 패턴이다. Ambiguity는 기준과 다른 예측이 가능한 사례의 비율이고, discrepancy는 기준과 가장 많이 다른 패턴의 사례 비율이다. 각 피처 조합은 COMPAS에서 나타난 횟수만큼 가중한다.

이 실험은 기록된 피처를 의도적으로 숨겼을 때의 변화를 같은 표본에서 측정한다. 관측 라벨은 그대로 사용하므로, 라벨이 깨끗한 조건에서 피처 누락만의 효과를 확인하는 실험은 앞의 합성 데이터 실험이다.

### 14. COMPAS에서 전과 피처 추가 비교

**하는 일과 이유**

같은 6,172개 사례의 관측 라벨과 깊이 2 이하 트리를 사용한다. 전과 피처의 제공 여부를 바꾸고 각 조건의 최고 정확도에서 0, 0.5, 1, 2%p 이내인 패턴을 모은다. 입력 조합은 해당 사례 수로 가중한다.

In [2]:
import csv
from collections import Counter
from pathlib import Path

compas_features = ("age_leq_25", "n_juvenile_felonies_geq_1", "n_priors_geq_1")
compas_path = Path("../data/compas_arrest_processed.csv")
if not compas_path.is_file():
    compas_path = Path("data/compas_arrest_processed.csv")

with compas_path.open(newline="") as file:
    compas_rows = [
        (tuple(int(row[feature]) for feature in compas_features), int(row["arrest"]) == 1)
        for row in csv.DictReader(file)
    ]

sample_size = len(compas_rows)
print("조건 | 최고 정확도 | 허용 오차(%p) | 패턴 수 | Ambiguity | Discrepancy")
for condition, feature_count in [("전과 피처 숨김", 2), ("전과 피처 포함", 3)]:
    label_counts = Counter((row[:feature_count], label) for row, label in compas_rows)
    groups = sorted({row for row, _ in label_counts})
    weights = [label_counts[row, 0] + label_counts[row, 1] for row in groups]
    patterns = tree_predictions(groups, range(feature_count), depth=2)
    correct_counts = {
        pattern: sum(label_counts[row, prediction] for row, prediction in zip(groups, pattern))
        for pattern in patterns
    }
    best_correct = max(correct_counts.values())
    reference = min(pattern for pattern, correct in correct_counts.items() if correct == best_correct)

    for tolerance_bp in (0, 50, 100, 200):
        near_optimal = [
            pattern for pattern, correct in correct_counts.items()
            if (best_correct - correct) * 10000 <= tolerance_bp * sample_size
        ]
        ambiguity = sum(
            weight for weight, predictions in zip(weights, zip(*near_optimal))
            if len(set(predictions)) > 1
        ) / sample_size
        discrepancy = max(
            sum(weight for weight, a, b in zip(weights, pattern, reference) if a != b)
            for pattern in near_optimal
        ) / sample_size
        print(
            f"{condition} | {best_correct / sample_size:.2%} | {tolerance_bp / 100:g} | "
            f"{len(near_optimal)} | {ambiguity:.2%} | {discrepancy:.2%}"
        )

조건 | 최고 정확도 | 허용 오차(%p) | 패턴 수 | Ambiguity | Discrepancy
전과 피처 숨김 | 58.07% | 0 | 1 | 0.00% | 0.00%
전과 피처 숨김 | 58.07% | 0.5 | 2 | 1.04% | 1.04%
전과 피처 숨김 | 58.07% | 1 | 3 | 3.37% | 2.33%
전과 피처 숨김 | 58.07% | 2 | 4 | 3.37% | 3.37%
전과 피처 포함 | 59.95% | 0 | 1 | 0.00% | 0.00%
전과 피처 포함 | 59.95% | 0.5 | 3 | 53.09% | 52.06%
전과 피처 포함 | 59.95% | 1 | 4 | 55.43% | 52.06%
전과 피처 포함 | 59.95% | 2 | 9 | 65.42% | 62.05%


**결과 읽기**

허용 오차 0은 정확히 최적인 패턴만 비교한다. 0.5%p에서는 최고 정확도보다 최대 0.5%p 낮은 패턴까지 포함한다. 전과 피처를 포함하면 이 범위의 ambiguity는 1.04%에서 53.09%로 늘고, 정확히 최적인 패턴은 두 조건에서 각각 하나다. 이 수치는 COMPAS의 관측 라벨을 사용한 표본 내 결과다.

### COMPAS 결과 해석

| 조건 | 표본 내 최고 정확도 | 최적 패턴 수 | 최고 정확도에서 0.5%p 이내의 패턴 수 | 0.5%p 이내 ambiguity |
|---|---:|---:|---:|---:|
| 전과 피처 숨김 | 58.07% | 1 | 2 | 1.04% |
| 전과 피처 포함 | 59.95% | 1 | 3 | 53.09% |

전과 피처를 포함했을 때 표본 내 최고 정확도와 근최적 패턴 간 예측 불일치가 함께 증가했다. 기존 두 피처가 모두 0이고 전과 피처가 1인 3,213개 사례에서 `+1`은 1,621개, `-1`은 1,592개였다. 이 집단의 예측을 뒤집으면 3,213개의 예측이 바뀌지만 정확도 차이는 29건, 약 0.47%p다.

두 조건 모두 정확히 최적인 예측 패턴은 한 개였다. 이 비교는 COMPAS의 관측 라벨에서 피처 하나를 숨겼다가 포함한 표본 내 결과다. 정답 규칙이 알려진 조건에서 피처 누락만으로 최적 예측이 여러 개 남는 경우는 앞의 합성 실험에서 확인했다.